In [1]:
import numpy as np
from IPython.display import clear_output
import time
import random
# https://www.youtube.com/watch?v=UXW2yZndl7U

In [2]:
# helper: convert a single cell to {-1,0,1}
def cell(board, r, c):
    # plus channel - minus channel
    return board[r, c, 0] - board[r, c, 1]

# helper: how many checkers are in a column (0..6)
def col_count(board, c):
    return int(np.sum(board[:, c, 0] + board[:, c, 1]))

In [3]:
def update_board(board_temp, color, column):
    # board_temp is (6,7,2)
    board = board_temp.copy()

    cnt = col_count(board, column)
    row = 5 - cnt  # lowest empty row index

    if row >= 0:
        if color == "plus":
            board[row, column, 0] = 1
            board[row, column, 1] = 0
        else:
            board[row, column, 1] = 1
            board[row, column, 0] = 0

    return board

In [4]:
# helper to display 6x7x2 board as 6x7 with values {+1,-1,0}
def board_to_2d(board_3d):
    return board_3d[:, :, 0] - board_3d[:, :, 1]

board = np.zeros((6, 7, 2), dtype=int)

board = update_board(board, 'plus', 3)
board = update_board(board, 'minus', 3)
board = update_board(board, 'plus', 3)
board = update_board(board, 'minus', 3)
board = update_board(board, 'plus', 3)
board = update_board(board, 'minus', 3)

print(board_to_2d(board))

[[ 0  0  0 -1  0  0  0]
 [ 0  0  0  1  0  0  0]
 [ 0  0  0 -1  0  0  0]
 [ 0  0  0  1  0  0  0]
 [ 0  0  0 -1  0  0  0]
 [ 0  0  0  1  0  0  0]]


In [5]:
def check_for_win_slow(board):
    nrow, ncol = 6, 7
    winner = "nobody"

    for col in range(ncol):
        for row in reversed(range(nrow)):
            if abs(cell(board, row, col)) < 0.1:
                break

            # vertical
            if row <= nrow - 4:
                tempsum = cell(board, row, col) + cell(board, row+1, col) + cell(board, row+2, col) + cell(board, row+3, col)
                if tempsum == 4:  return "v-plus"
                if tempsum == -4: return "v-minus"

            # horizontal
            if col <= ncol - 4:
                tempsum = cell(board, row, col) + cell(board, row, col+1) + cell(board, row, col+2) + cell(board, row, col+3)
                if tempsum == 4:  return "h-plus"
                if tempsum == -4: return "h-minus"

            # diag down-right
            if (row <= nrow - 4) and (col <= ncol - 4):
                tempsum = cell(board, row, col) + cell(board, row+1, col+1) + cell(board, row+2, col+2) + cell(board, row+3, col+3)
                if tempsum == 4:  return "d-plus"
                if tempsum == -4: return "d-minus"

            # diag down-left
            if (row <= nrow - 4) and (col >= 3):
                tempsum = cell(board, row, col) + cell(board, row+1, col-1) + cell(board, row+2, col-2) + cell(board, row+3, col-3)
                if tempsum == 4:  return "d-plus"
                if tempsum == -4: return "d-minus"

    return winner

In [6]:
def check_for_win(board, col):
    # faster version; requires last played column
    nrow, ncol = 6, 7

    cnt = col_count(board, col)
    row = int(6 - cnt)  # same logic as original code

    # all sums use cell(board, r, c)
    if row + 3 < 6:
        vert = cell(board, row, col) + cell(board, row+1, col) + cell(board, row+2, col) + cell(board, row+3, col)
        if vert == 4:  return "v-plus"
        if vert == -4: return "v-minus"

    # horizontals (same pattern as original)
    if col + 3 < 7:
        hor = cell(board, row, col) + cell(board, row, col+1) + cell(board, row, col+2) + cell(board, row, col+3)
        if hor == 4:  return "h-plus"
        if hor == -4: return "h-minus"

    if col - 1 >= 0 and col + 2 < 7:
        hor = cell(board, row, col-1) + cell(board, row, col) + cell(board, row, col+1) + cell(board, row, col+2)
        if hor == 4:  return "h-plus"
        if hor == -4: return "h-minus"

    if col - 2 >= 0 and col + 1 < 7:
        hor = cell(board, row, col-2) + cell(board, row, col-1) + cell(board, row, col) + cell(board, row, col+1)
        if hor == 4:  return "h-plus"
        if hor == -4: return "h-minus"

    if col - 3 >= 0:
        hor = cell(board, row, col-3) + cell(board, row, col-2) + cell(board, row, col-1) + cell(board, row, col)
        if hor == 4:  return "h-plus"
        if hor == -4: return "h-minus"

    # diagonals down-right
    if row < 3 and col < 4:
        dr = cell(board, row, col) + cell(board, row+1, col+1) + cell(board, row+2, col+2) + cell(board, row+3, col+3)
        if dr == 4:  return "d-plus"
        if dr == -4: return "d-minus"

    if row-1 >= 0 and col-1 >= 0 and row+2 < 6 and col+2 < 7:
        dr = cell(board, row-1, col-1) + cell(board, row, col) + cell(board, row+1, col+1) + cell(board, row+2, col+2)
        if dr == 4:  return "d-plus"
        if dr == -4: return "d-minus"

    if row-2 >= 0 and col-2 >= 0 and row+1 < 6 and col+1 < 7:
        dr = cell(board, row-2, col-2) + cell(board, row-1, col-1) + cell(board, row, col) + cell(board, row+1, col+1)
        if dr == 4:  return "d-plus"
        if dr == -4: return "d-minus"

    if row-3 >= 0 and col-3 >= 0:
        dr = cell(board, row-3, col-3) + cell(board, row-2, col-2) + cell(board, row-1, col-1) + cell(board, row, col)
        if dr == 4:  return "d-plus"
        if dr == -4: return "d-minus"

    # diagonals down-left
    if row + 3 < 6 and col - 3 >= 0:
        dl = cell(board, row, col) + cell(board, row+1, col-1) + cell(board, row+2, col-2) + cell(board, row+3, col-3)
        if dl == 4:  return "d-plus"
        if dl == -4: return "d-minus"

    if row-1 >= 0 and col+1 < 7 and row+2 < 6 and col-2 >= 0:
        dl = cell(board, row-1, col+1) + cell(board, row, col) + cell(board, row+1, col-1) + cell(board, row+2, col-2)
        if dl == 4:  return "d-plus"
        if dl == -4: return "d-minus"

    if row-2 >= 0 and col+2 < 7 and row+1 < 6 and col-1 >= 0:
        dl = cell(board, row-2, col+2) + cell(board, row-1, col+1) + cell(board, row, col) + cell(board, row+1, col-1)
        if dl == 4:  return "d-plus"
        if dl == -4: return "d-minus"

    if row-3 >= 0 and col+3 < 7:
        dl = cell(board, row-3, col+3) + cell(board, row-2, col+2) + cell(board, row-1, col+1) + cell(board, row, col)
        if dl == 4:  return "d-plus"
        if dl == -4: return "d-minus"

    return "nobody"

In [7]:
def find_legal(board):
    # legal if top cell is empty (both channels 0)
    legal = [i for i in range(7) if (board[0, i, 0] + board[0, i, 1]) < 0.1]
    return legal

In [8]:
def look_for_win(board_,color):
    board_ = board_.copy()
    legal = find_legal(board_)
    winner = -1
    for m in legal:
        bt = update_board(board_.copy(),color,m)
        wi = check_for_win(bt,m)
        if wi[2:] == color:
            winner = m
            break
    return winner

In [9]:
def find_all_nonlosers(board,color):
    if color == 'plus':
        opp = 'minus'
    else:
        opp = 'plus'
    legal = find_legal(board)
    poss_boards = [update_board(board,color,l) for l in legal]
    poss_legal = [find_legal(b) for b in poss_boards]
    allowed = []
    for i in range(len(legal)):
        wins = [j for j in poss_legal[i] if check_for_win(update_board(poss_boards[i],opp,j),j) != 'nobody']
        if len(wins) == 0:
            allowed.append(legal[i])
    return allowed

In [10]:
def back_prop(winner,path,color0,md):
    for i in range(len(path)):
        board_temp = path[i]

        md[board_temp][0]+=1
        if winner[2]==color0[0]:
            if i % 2 == 1:
                md[board_temp][1] += 1
            else:
                md[board_temp][1] -= 1
        elif winner[2]=='e': # tie
            # md[board_temp][1] += 0
            pass
        else:
            if i % 2 == 1:
                md[board_temp][1] -= 1
            else:
                md[board_temp][1] += 1

In [11]:
def rollout(board,next_player):
    winner = 'nobody'
    player = next_player
    while winner == 'nobody':
        legal = find_legal(board)
        if len(legal) == 0:
            winner = 'tie'
            return winner
        move = random.choice(legal)
        board = update_board(board,player,move)
        winner = check_for_win(board,move)

        if player == 'plus':
            player = 'minus'
        else:
            player = 'plus'
    return winner


In [12]:
def mcts(board_temp,color0,time_limit):
    # time_limit is a parameter that determines the skill (and slowness) of the player
    # bigger values of time_limit means the player is better, but also slower to figure out a move.
    board = board_temp.copy()
    ##############################################
    winColumn = look_for_win(board,color0) # check to find a winning column
    if winColumn > -0.5:
        return winColumn, 0 # if there is one - play that! (0 iterations as it's an immediate win)
    legal0 = find_all_nonlosers(board,color0) # find all moves that won't immediately lead to your opponent winning
    if len(legal0) == 0: # if you can't block your opponent - just find the 'best' losing move
        legal0 = find_legal(board)
    ##############################################
    # the code above, in between the hash rows, is not part of traditional MCTS
    # but it makes it better and faster - so I included it!
    # MCTS occasionally makes stupid mistakes
    # like not dropping the checker on a winning column, or not blocking an obvious opponent win
    # this avoids a little bit of that stupidity!
    # we could also add this logic to the rest of the MCTS and rollout functions - I just haven't done that yet...
    # feel free to experiment!

    mcts_dict = {tuple(board.ravel()):[0,0]}
    start_time = time.time()
    iterations_performed = 0

    while time.time() - start_time < time_limit:
        iterations_performed += 1
        color = color0
        winner = 'nobody'
        board_mcts = board.copy()
        path = [tuple(board_mcts.ravel())]
        while winner == 'nobody':
            legal = find_legal(board_mcts)
            if len(legal) == 0:
                winner = 'tie'
                back_prop(winner,path,color0,mcts_dict)
                break
            board_list = []
            for col in legal:
                board_list.append(tuple(update_board(board_mcts,color,col).ravel()))
            for bl in board_list:
                if bl not in mcts_dict.keys():
                    mcts_dict[bl] = [0,0]
            ucb1 = np.zeros(len(legal))
            for i in range(len(legal)):
                num_denom = mcts_dict[board_list[i]]
                if num_denom[0] == 0:
                    ucb1[i] = 10*time_limit # Use a large value to prioritize unvisited nodes
                else:
                    # UCB1 formula: exploit + explore
                    ucb1[i] = num_denom[1]/num_denom[0] + 2*np.sqrt(np.log(mcts_dict[path[-1]][0])/mcts_dict[board_list[i]][0])
            chosen = np.argmax(ucb1)

            board_mcts = update_board(board_mcts,color,legal[chosen])
            path.append(tuple(board_mcts.ravel()))
            winner = check_for_win(board_mcts,legal[chosen])
            if winner[2]==color[0]:
                back_prop(winner,path,color0,mcts_dict)
                break
            if color == 'plus':
                color = 'minus'
            else:
                color = 'plus'
            if mcts_dict[tuple(board_mcts.ravel())][0] == 0:
                winner = rollout(board_mcts,color)
                back_prop(winner,path,color0,mcts_dict)
                break

    maxval = -np.inf
    best_col = -1
    # Ensure legal0 is not empty, fall back to find_legal if necessary
    if not legal0:
        legal0 = find_legal(board)
    if not legal0: # Should not happen unless board is full already
        return -1, iterations_performed

    for col in legal0:
        board_after_move = update_board(board, color0, col)
        board_temp = tuple(board_after_move.ravel())
        num_denom = mcts_dict.get(board_temp, [0, 0]) # Use .get with default to handle cases where a board might not be in the dict
        if num_denom[0] == 0:
            compare = -np.inf
        else:
            compare = num_denom[1] / num_denom[0]
        if compare > maxval:
            maxval = compare
            best_col = col

    if best_col == -1 and legal0: # Fallback if no specific best_col was found but legal moves exist
        best_col = legal0[0]

    return (best_col, iterations_performed)

In [13]:
mcts(np.zeros((6, 7, 2)), 'plus', 1)

(4, 562)

In [14]:
import numpy as np

def board_to_2d(board_3d):
    return board_3d[:, :, 0] - board_3d[:, :, 1]

board = np.zeros((6, 7, 2), dtype=int)
winner = 'nobody'
color = 'plus'

while winner == 'nobody':
    if color == 'minus':
        col, iters = mcts(board, color, 5)   # 5 seconds
    else:
        col, iters = mcts(board, color, 10)  # 10 seconds

    # safety: ensure col is valid
    legal = find_legal(board)
    if col not in legal:
        if len(legal) == 0:
            winner = 'tie'
            break
        col = legal[0]

    board = update_board(board, color, col)
    winner = check_for_win(board, col)

    print(f"player={color} move={col} iters={iters}")
    print(board_to_2d(board))
    print('=========================')

    color = 'minus' if color == 'plus' else 'plus'

print(winner)

player=plus move=3 iters=4311
[[0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 1 0 0 0]]
player=minus move=2 iters=2385
[[ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0 -1  1  0  0  0]]
player=plus move=3 iters=4249
[[ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  1  0  0  0]
 [ 0  0 -1  1  0  0  0]]
player=minus move=3 iters=2257
[[ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0 -1  0  0  0]
 [ 0  0  0  1  0  0  0]
 [ 0  0 -1  1  0  0  0]]
player=plus move=2 iters=4569
[[ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0 -1  0  0  0]
 [ 0  0  1  1  0  0  0]
 [ 0  0 -1  1  0  0  0]]
player=minus move=2 iters=2474
[[ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0 -1 -1  0  0  0]
 [ 0  0  1  1  0  0  0]
 [ 0  0 -1  1 

In [15]:
def display_board(board):
    # this function displays the board as ascii using X for +1 and O for -1
    # For the project, this should be a better picture of the board...
    clear_output()
    horizontal_line = '-'*(7*5+8)
    blank_line = '|'+' '*5
    blank_line *= 7
    blank_line += '|'
    print('   0     1     2     3     4     5     6')
    print(horizontal_line)
    for row in range(6):
        print(blank_line)
        this_line = '|'
        for col in range(7):
            if board[row,col] == 0:
                this_line += ' '*5 + '|'
            elif board[row,col] == 1:
                this_line += '  X  |'
            else:
                this_line += '  O  |'
        print(this_line)
        print(blank_line)
        print(horizontal_line)
    print('   0     1     2     3     4     5     6')

In [16]:
def board_to_2d(board_3d):
    return board_3d[:, :, 0] - board_3d[:, :, 1]

# wrapper so your existing display_board can stay the same
def display_board_3d(board_3d):
    display_board(board_to_2d(board_3d))

In [20]:
# this is how you can play a game
winner = 'nobody'
board = np.zeros((6, 7, 2), dtype=int)

display_board_3d(board)

player = 'plus'

while winner == 'nobody':
    move = input('Pick a move (0-6) for player ' + player + ': ')
    move = int(move)

    board = update_board(board, player, move)
    display_board_3d(board)

    winner = check_for_win(board, move)

    player = 'minus' if player == 'plus' else 'plus'

print('The winner is ' + winner)

   0     1     2     3     4     5     6
-------------------------------------------
|     |     |     |     |     |     |     |
|     |     |     |     |     |     |     |
|     |     |     |     |     |     |     |
-------------------------------------------
|     |     |     |     |     |     |     |
|     |     |     |     |     |     |     |
|     |     |     |     |     |     |     |
-------------------------------------------
|     |     |     |     |     |     |     |
|     |     |     |     |     |     |     |
|     |     |     |     |     |     |     |
-------------------------------------------
|     |     |     |     |     |     |     |
|     |     |     |     |     |     |     |
|     |     |     |     |     |     |     |
-------------------------------------------
|     |     |     |     |     |     |     |
|     |     |     |     |     |     |     |
|     |     |     |     |     |     |     |
-------------------------------------------
|     |     |     |     |     |    

KeyboardInterrupt: Interrupted by user

# **Build dataset**

In [ ]:
def to_current_player_perspective(board, player):
    """
    Ensure channel 0 always represents the CURRENT player to move,
    and channel 1 represents the opponent.
    """
    if player == "plus":
        return board
    b = board.copy()
    b[:, :, [0, 1]] = b[:, :, [1, 0]]  # swap channels
    return b

def generate_dataset_target_states(target_states, time_limit, random_opening_moves, seed=0, save_prefix=''):
    rng = random.Random(seed)

    X = []
    y = []
    states_collected = 0
    games_played = 0

    while states_collected < target_states:
        board = np.zeros((6, 7, 2), dtype=np.int8)
        winner = "nobody"
        player = "plus"
        moves_played = 0
        games_played += 1

        while winner == "nobody" and states_collected < target_states:
            legal = find_legal(board)
            if len(legal) == 0:
                break  # tie / full board

            # random opening moves for diversity (do NOT label/save these)
            if moves_played < random_opening_moves:
                move = rng.choice(legal)
            else:
                move, iters = mcts(board, player, time_limit)  # <-- unpack here

                # safety: if MCTS returns illegal, fall back to random legal
                if move not in legal:
                    move = rng.choice(legal)

                # SAVE training example BEFORE applying the move
                X.append(to_current_player_perspective(board, player))
                y.append(move)
                states_collected += 1

            board = update_board(board, player, move)
            winner = check_for_win(board, move)

            player = "minus" if player == "plus" else "plus"
            moves_played += 1

    X = np.stack(X).astype(np.int8)     # (N, 6, 7, 2)
    y = np.array(y, dtype=np.int64)     # (N,)

    np.save(f"{save_prefix}_X_boards.npy", X)
    np.save(f"{save_prefix}_y_moves.npy", y)

    print("Saved:", f"{save_prefix}_X_boards.npy", f"{save_prefix}_y_moves.npy")
    print("X shape:", X.shape, "dtype:", X.dtype)
    print("y shape:", y.shape, "dtype:", y.dtype)

    return X, y

In [ ]:
#X_boards, y_moves = generate_dataset_target_states(target_states=100, nsteps=800, random_opening_moves=2, seed=0)

#print(X_boards.shape, y_moves.shape)
#print(X_boards.dtype, y_moves.dtype)

# **Generate more data..**

In [ ]:
X, y = generate_dataset_target_states(target_states=10000, time_limit=1, random_opening_moves=0, seed=2, save_prefix="h10k_t1")


Saved: h10k_t1_X_boards.npy h10k_t1_y_moves.npy
X shape: (10000, 6, 7, 2) dtype: int8
y shape: (10000,) dtype: int64


# **Save data files**

In [ ]:
!mv c10k_t1_X_boards.npy /content/drive/MyDrive/optimization/connect4/connect4_optimization/
!mv c10k_t1_y_moves.npy /content/drive/MyDrive/optimization/connect4/connect4_optimization/

mv: cannot move 'c10k_t1_X_boards.npy' to '/content/drive/MyDrive/optimization/connect4/connect4_optimization/': No such file or directory
mv: cannot move 'c10k_t1_y_moves.npy' to '/content/drive/MyDrive/optimization/connect4/connect4_optimization/': No such file or directory


In [17]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [18]:
import os
DATA_DIR = "/content/drive/MyDrive/optimization/connect4/connect4_optimization/data"

print("\n".join(sorted(os.listdir(DATA_DIR))))

X_boards.npy
a10k_t1_X_boards.npy
a10k_t1_y_moves.npy
b10k_t1_X_boards.npy
b10k_t1_y_moves.npy
c10k_t1_X_boards.npy
c10k_t1_y_moves.npy
c4_10k_s200_X_boards.npy
c4_10k_s200_y_moves.npy
c4_10k_t02_1_X_boards.npy
c4_10k_t02_1_y_moves.npy
c4_10k_t02_2_X_boards.npy
c4_10k_t02_2_y_moves.npy
c4_10k_t02_3_X_boards.npy
c4_10k_t02_3_y_moves.npy
c4_10k_t02_X_boards.npy
c4_10k_t02_y_moves.npy
c4_10k_t1_X_boards.npy
c4_10k_t1_y_moves.npy
c4_10k_t2_1_X_boards.npy
c4_10k_t2_1_y_moves.npy
c4_1_10k_s200_X_boards.npy
c4_1_10k_s200_y_moves.npy
c4_1k_X_boards.npy
c4_1k_y_moves.npy
c4_20k_s400_X_boards.npy
c4_20k_s400_y_moves.npy
c4_20k_s600_X_boards.npy
c4_20k_s600_y_moves.npy
c4_20k_t02_1_X_boards.npy
c4_20k_t02_1_y_moves.npy
c4_20k_t02_X_boards.npy
c4_20k_t02_y_moves.npy
c4_20k_t2_X_boards.npy
c4_20k_t2_y_moves.npy
c4_5k_t1_X_boards.npy
c4_5k_t1_y_moves.npy
c4_5k_t2_X_boards.npy
c4_5k_t2_y_moves.npy
d10k_t1_X_boards.npy
d10k_t1_y_moves.npy
e10k_t1_X_boards.npy
e10k_t1_y_moves.npy
f10k_t1_X_boards.npy
f

# **Combine all the data files**

In [20]:
import os
import numpy as np

DATA_DIR = "/content/drive/MyDrive/optimization/connect4/connect4_optimization/data"

PAIRS = [
    ("a10k_t1_X_boards.npy", "a10k_t1_y_moves.npy"),
    ("b10k_t1_X_boards.npy", "b10k_t1_y_moves.npy"),
    ("c10k_t1_X_boards.npy", "c10k_t1_y_moves.npy"),
    ("c4_10k_t02_1_X_boards.npy", "c4_10k_t02_1_y_moves.npy"),
    ("c4_10k_t02_2_X_boards.npy", "c4_10k_t02_2_y_moves.npy"),
    ("c4_10k_t02_3_X_boards.npy", "c4_10k_t02_3_y_moves.npy"),
    ("c4_10k_t1_X_boards.npy", "c4_10k_t1_y_moves.npy"),
    ("c4_10k_t2_1_X_boards.npy", "c4_10k_t2_1_y_moves.npy"),
    ("c4_1k_X_boards.npy", "c4_1k_y_moves.npy"),
    ("c4_20k_s400_X_boards.npy", "c4_20k_s400_y_moves.npy"),
    ("c4_20k_s600_X_boards.npy", "c4_20k_s600_y_moves.npy"),
    ("c4_20k_t02_1_X_boards.npy", "c4_20k_t02_1_y_moves.npy"),
    ("c4_20k_t02_X_boards.npy", "c4_20k_t02_y_moves.npy"),
    ("c4_20k_t2_X_boards.npy", "c4_20k_t2_y_moves.npy"),
    ("c4_5k_t1_X_boards.npy", "c4_5k_t1_y_moves.npy"),
    ("c4_5k_t2_X_boards.npy", "c4_5k_t2_y_moves.npy"),
    ("d10k_t1_X_boards.npy", "d10k_t1_y_moves.npy"),
    ("e10k_t1_X_boards.npy", "e10k_t1_y_moves.npy"),
    ("f10k_t1_X_boards.npy", "f10k_t1_y_moves.npy"),
    ("g10k_t1_X_boards.npy", "g10k_t1_y_moves.npy"),
    ("h10k_t1_X_boards.npy", "h10k_t1_y_moves.npy"),
    ("mcts7500_X.npy", "mcts7500_y.npy")
]

assert len(PAIRS) == 22

Xs, ys = [], []

for x_name, y_name in PAIRS:
    X = np.load(os.path.join(DATA_DIR, x_name))
    y = np.load(os.path.join(DATA_DIR, y_name))

    assert len(X) == len(y), f"Length mismatch: {x_name}"
    assert X.shape[1:] == (6,7,2), f"Bad shape in {x_name}: {X.shape}"
    assert y.min() >= 0 and y.max() <= 6, f"Bad labels in {y_name}"

    Xs.append(X)
    ys.append(y)

X_all = np.concatenate(Xs, axis=0)
y_all = np.concatenate(ys, axis=0)

print("Combined dataset:")
print("X:", X_all.shape, X_all.dtype)
print("y:", y_all.shape, y_all.dtype)
print("Label range:", y_all.min(), y_all.max())

Combined dataset:
X: (516620, 6, 7, 2) int16
y: (516620,) int64
Label range: 0 6


# **Double the dataset by flipping over y-axis**

In [21]:
X_flip = X_all[:, :, ::-1, :]   # flip columns
y_flip = 6 - y_all              # flip move labels

X_aug = np.concatenate([X_all, X_flip], axis=0)
y_aug = np.concatenate([y_all, y_flip], axis=0)

print("Augmented shape:", X_aug.shape, y_aug.shape)

Augmented shape: (1033240, 6, 7, 2) (1033240,)


In [22]:
perm = np.random.permutation(len(X_aug))
X_aug = X_aug[perm]
y_aug = y_aug[perm]

Sanity checks...

In [24]:
# shapes
print(X_aug.shape, y_aug.shape)

# labels still valid
print(y_aug.min(), y_aug.max())  # should be 0..6

# no overlapping channels
assert np.all((X_aug[:,:,:,0] + X_aug[:,:,:,1]) <= 1)

# test a random flip manually
i = np.random.randint(len(X_all))
print("original label:", y_all[i])
print("flipped label:", y_flip[i])

(1033240, 6, 7, 2) (1033240,)
0 6
original label: 1
flipped label: 5


# **Poorly classified boards**

In [ ]:
from collections import defaultdict, Counter
import numpy as np

X_flat = X_all.reshape(len(X_all), -1)
board_counts = defaultdict(Counter)

for b, y in zip(X_flat, y_all):
    board_counts[tuple(b)][int(y)] += 1

# compute agreement scores
records = []

for board, counter in board_counts.items():
    total = sum(counter.values())
    if total > 1:
        best = counter.most_common(1)[0][1]
        agreement = best / total
        records.append((agreement, total, counter, board))

# sort by worst agreement
records.sort(key=lambda x: x[0])

print("Worst 10 boards by agreement:\n")
for agr, total, counter, _ in records[:10]:
    print(f"agreement={agr:.2f}  total={total}  labels={dict(counter)}")

Worst 10 boards by agreement:

agreement=0.25  total=4  labels={3: 1, 6: 1, 2: 1, 4: 1}
agreement=0.25  total=4  labels={5: 1, 3: 1, 2: 1, 6: 1}
agreement=0.25  total=4  labels={5: 1, 6: 1, 2: 1, 3: 1}
agreement=0.27  total=15  labels={6: 2, 0: 1, 3: 4, 2: 3, 5: 4, 4: 1}
agreement=0.28  total=36  labels={2: 7, 3: 6, 5: 7, 6: 6, 4: 10}
agreement=0.29  total=7  labels={2: 2, 4: 2, 1: 1, 5: 2}
agreement=0.30  total=27  labels={5: 7, 2: 6, 3: 6, 1: 8}
agreement=0.32  total=558  labels={4: 114, 3: 179, 2: 156, 0: 16, 5: 44, 1: 42, 6: 7}
agreement=0.33  total=3  labels={2: 1, 1: 1, 5: 1}
agreement=0.33  total=3  labels={5: 1, 3: 1, 1: 1}


In [ ]:
def show_board_from_flat(flat):
    b = np.array(flat, dtype=np.int8).reshape(6,7,2)
    grid = b[:,:,0] - b[:,:,1]
    print(grid)

print("Top 5 most inconsistent boards:\n")

for i in range(5):
    agreement, total, counter, board = records[i]

    print("="*40)
    print(f"Rank #{i+1}")
    print(f"Agreement: {agreement:.2f}")
    print(f"Occurrences: {total}")
    print(f"Label counts: {dict(counter)}")
    print("Board:")
    show_board_from_flat(board)

Top 5 most inconsistent boards:

Rank #1
Agreement: 0.25
Occurrences: 4
Label counts: {3: 1, 6: 1, 2: 1, 4: 1}
Board:
[[ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  1  0 -1  0  0]]
Rank #2
Agreement: 0.25
Occurrences: 4
Label counts: {5: 1, 3: 1, 2: 1, 6: 1}
Board:
[[ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0 -1  1  0  0  0]
 [ 0  1 -1 -1  0  0  1]]
Rank #3
Agreement: 0.25
Occurrences: 4
Label counts: {5: 1, 6: 1, 2: 1, 3: 1}
Board:
[[ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0 -1  1  0 -1  1]]
Rank #4
Agreement: 0.27
Occurrences: 15
Label counts: {6: 2, 0: 1, 3: 4, 2: 3, 5: 4, 4: 1}
Board:
[[ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0]
 [ 0  0 -1 -1  0  0  0]
 [ 0  0  1  1  0  0  0]
 [ 0  0 -1  1  0  0  0]]
Rank #5
Agreement: 0.28
Occurrence

# **Apply majority-vote move selection for duplicated boards**

In [25]:
X_flat = X_aug.reshape(len(X_aug), -1)

unique_boards, counts = np.unique(X_flat, axis=0, return_counts=True)

num_total = len(X_aug)
num_unique = len(unique_boards)
num_duplicates = num_total - num_unique

print(f"Total boards:     {num_total}")
print(f"Unique boards:    {num_unique}")
print(f"Duplicate boards: {num_duplicates}")

Total boards:     1033240
Unique boards:    873640
Duplicate boards: 159600


In [26]:
from collections import defaultdict, Counter

X_flat = X_aug.reshape(len(X_aug), -1)

# Map: board -> Counter of labels
board_label_counts = defaultdict(Counter)

for board, label in zip(X_flat, y_aug):
    board_label_counts[tuple(board)][int(label)] += 1

# Build deduplicated dataset
X_dedup = []
y_dedup = []

for board_tuple, label_counter in board_label_counts.items():
    # choose most common label
    best_label = label_counter.most_common(1)[0][0]
    X_dedup.append(np.array(board_tuple, dtype=np.int8).reshape(6, 7, 2))
    y_dedup.append(best_label)

X_dedup = np.stack(X_dedup)
y_dedup = np.array(y_dedup, dtype=np.int64)

print("After deduplication:")
print("X:", X_dedup.shape, X_dedup.dtype)
print("y:", y_dedup.shape, y_dedup.dtype)
print("Label range:", y_dedup.min(), y_dedup.max())

After deduplication:
X: (873640, 6, 7, 2) int8
y: (873640,) int64
Label range: 0 6


In [27]:
np.save("X_final.npy", X_dedup)
np.save("y_final.npy", y_dedup)

# **Test .npy files**

### Collapsing 2 channels into 1 for visualization
* Channel 0 is ALWAYS the current player to move.
Channel 1 is ALWAYS the opponent.
* +1 = current player  |  -1 = opponent

In [ ]:
X = np.load("X_boards.npy")
y = np.load("y_moves.npy")

i = np.random.randint(len(X))
board = X[i]
move = y[i]

print("Label move:", move)
print(board[:, :, 0] - board[:, :, 1])

Label move: 0
[[ 0  0  0 -1  0  0  0]
 [ 0  0  0  1 -1  0  0]
 [ 1  0  0  1  1  0 -1]
 [-1  0  0 -1 -1  0  1]
 [-1  0  0  1 -1  0  1]
 [-1 -1  0  1 -1  1  1]]


### Seeing both channels, not collapsing

In [ ]:
print("Label move:", move)

print("\nCurrent player (ch 0) | Opponent (ch 1)")
for r in range(6):
    left = board[r, :, 0]
    right = board[r, :, 1]
    print(f"{left}         {right}")

Label move: 0

Current player (ch 0) | Opponent (ch 1)
[0 0 0 0 0 0 0]         [0 0 0 1 0 0 0]
[0 0 0 1 0 0 0]         [0 0 0 0 1 0 0]
[1 0 0 1 1 0 0]         [0 0 0 0 0 0 1]
[0 0 0 0 0 0 1]         [1 0 0 1 1 0 0]
[0 0 0 1 0 0 1]         [1 0 0 0 1 0 0]
[0 0 0 1 0 1 1]         [1 1 0 0 1 0 0]


In [1]:
from google.colab import files
import pickle

# Upload file from your computer
uploaded = files.upload()

# Read pickle file
with open("mcts7500_pool.pickle", "rb") as f:
    data = pickle.load(f)

print(type(data))

Saving mcts7500_pool.pickle to mcts7500_pool.pickle


/tmp/ipython-input-1998894660.py:9: DeprecationWarning: numpy.core.numeric is deprecated and has been renamed to numpy._core.numeric. The numpy._core namespace contains private NumPy internals and its use is discouraged, as NumPy internals can change without warning in any release. In practice, most real-world usage of numpy.core is to access functionality in the public NumPy API. If that is the case, use the public NumPy API. If not, you are using NumPy internals. If you would still like to access an internal attribute, use numpy._core.numeric._frombuffer.
  data = pickle.load(f)


<class 'dict'>


In [13]:
import numpy as np

# data is already loaded from pickle
boards = data["board_x"]
moves  = data["play_y"]

# Convert to arrays
boards = np.asarray(boards)          # (N, 6, 7)
moves  = np.asarray(moves, dtype=np.int64)  # (N,)

# 6x7x2 encoding: [current_player(+1), opponent(-1)]
X = np.zeros((boards.shape[0], 6, 7, 2), dtype=np.uint8)
X[..., 0] = (boards ==  1)  # + pieces
X[..., 1] = (boards == -1)  # - pieces

In [16]:
# Save
np.save("mcts7500_X.npy", X)
np.save("mcts7500_y.npy", moves)

In [14]:
len(X)

265620

In [15]:
len(moves)

265620

In [19]:
print(X.shape)        # should be (N, 6, 7, 2)
print(moves.shape)        # should be (N,)

(265620, 6, 7, 2)
(265620,)


In [10]:
idx = 0

print("Original board:")
print(boards[idx])

print("\nChannel 0 (+ pieces):")
print(X[idx, :, :, 0])

print("\nChannel 1 (- pieces):")
print(X[idx, :, :, 1])

print("\nCorresponding y move (column to play):")
print(moves[idx])

Original board:
[[ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  1. -1.  1.]
 [-1.  1.  0.  0.  1. -1. -1.]]

Channel 0 (+ pieces):
[[0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 0 1 0 1]
 [0 1 0 0 1 0 0]]

Channel 1 (- pieces):
[[0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0]
 [0 0 0 0 0 1 0]
 [1 0 0 0 0 1 1]]

Corresponding y move (column to play):
4
